In [1]:
from langchain_community.document_loaders import  PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [2]:
load_dotenv()
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

db_name = "./chroma_vector_store_crag"

In [ ]:
import uuid
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

def ingest_document(file_path: str):
    try:
        loader = PyPDFLoader(file_path)
        docs = loader.load()

        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        chunks = splitter.split_documents(docs)

        for chunk in chunks:
            chunk.metadata["source"] = file_path
            chunk.metadata["page"] = chunk.metadata.get("page", 0)

        ids = [str(uuid.uuid4()) for _ in chunks]

        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            ids=ids,
            persist_directory=db_name
        )
        return f"Success: Ingested {len(chunks)} chunks from {file_path} into the vector database."
    except Exception as e:
        return f"Error ingesting file: {str(e)}"

In [4]:
ingest_document("data/agenticAi.pdf")

'Success: Ingested 39 chunks from data/agenticAi.pdf into the vector database.'